# Sentiment Analysis — Full ML Pipeline
> **3-class** (negative / neutral / positive) · Classical ML + DistilBERT fine-tuning · Plug-and-play dataset config

In [ ]:
import importlib, subprocess, sys

required = {
    'datasets':       'datasets',
    'transformers':   'transformers',
    'torch':          'torch',
    'sklearn':        'scikit-learn',
    'wordcloud':      'wordcloud',
    'seaborn':        'seaborn',
    'accelerate':     'accelerate',
}

missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f'Installing: {missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    print('Done. Restart the kernel if this is the first install.')
else:
    print('All dependencies present.')

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False
    print('wordcloud not installed — word cloud cell will be skipped.\n'
          'Install with: pip install wordcloud')

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from datasets import load_dataset
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset as TorchDataset

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Configuration
**All dataset/model settings live here.** Swap values to run on any other dataset.

In [ ]:
# ── Data Source ───────────────────────────────────────────────────────────────
DATA_SOURCE    = 'huggingface'   # 'huggingface' | 'csv'
DATASET_NAME   = 'Sp1786/multiclass-sentiment-analysis-dataset'
DATASET_CONFIG = None            # set to subset name if needed (e.g. 'sentiment')
CSV_PATH       = None            # used when DATA_SOURCE='csv'

TEXT_COL  = 'text'
LABEL_COL = 'label'
LABEL_MAP = {0: 'negative', 1: 'neutral', 2: 'positive'}
# ─────────────────────────────────────────────────────────────────────────────

MAX_SAMPLES  = 6000   # cap total samples for speed; set None for full ~41k dataset
TEST_SIZE    = 0.20
RANDOM_STATE = 42

# ── Transformer ───────────────────────────────────────────────────────────────
BERT_MODEL   = 'distilbert-base-uncased'
BERT_MAX_LEN = 128
BERT_EPOCHS  = 3
BERT_BATCH   = 16
BERT_SAMPLES = 4000   # training samples used for fine-tuning; set None for all
# ─────────────────────────────────────────────────────────────────────────────

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Data Loading

In [ ]:
if DATA_SOURCE == 'huggingface':
    raw = load_dataset(DATASET_NAME) if not DATASET_CONFIG else load_dataset(DATASET_NAME, DATASET_CONFIG)
    dfs = [raw[s].to_pandas()[[TEXT_COL, LABEL_COL]] for s in raw.keys()]
    df  = pd.concat(dfs, ignore_index=True)
else:
    assert CSV_PATH, 'Set CSV_PATH in Config when DATA_SOURCE="csv"'
    df = pd.read_csv(CSV_PATH) if CSV_PATH.endswith('.csv') else pd.read_excel(CSV_PATH)
    if df[LABEL_COL].dtype == object:                    # string labels → int
        inv_map = {v: k for k, v in LABEL_MAP.items()}
        df[LABEL_COL] = df[LABEL_COL].map(inv_map)

df = (df.dropna(subset=[TEXT_COL, LABEL_COL])
        .drop_duplicates(subset=[TEXT_COL])
        .reset_index(drop=True))

if MAX_SAMPLES:
    per_class = MAX_SAMPLES // len(LABEL_MAP)
    df = (df.groupby(LABEL_COL, group_keys=False)
            .apply(lambda x: x.sample(min(len(x), per_class), random_state=RANDOM_STATE))
            .reset_index(drop=True))

df['label_name'] = df[LABEL_COL].map(LABEL_MAP)
print(f'Loaded {len(df):,} samples | Classes: {df[LABEL_COL].nunique()}')
df.head(3)

## 3. Exploratory Data Analysis

In [ ]:
COLORS = ['#e74c3c', '#95a5a6', '#2ecc71']
df['text_len'] = df[TEXT_COL].str.split().str.len()
counts = df['label_name'].value_counts().reindex(LABEL_MAP.values())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(counts.index, counts.values, color=COLORS)
axes[0].set_title('Class Distribution'); axes[0].set_ylabel('Count')

for col, lbl in zip(COLORS, LABEL_MAP.values()):
    axes[1].hist(df[df['label_name'] == lbl]['text_len'], bins=30, alpha=0.6, label=lbl, color=col)
axes[1].set_title('Text Length (words)'); axes[1].legend()

axes[2].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=COLORS)
axes[2].set_title('Class Balance')

plt.tight_layout(); plt.show()
print(df['text_len'].describe().round(2).to_string())

In [ ]:
if not HAS_WORDCLOUD:
    print('Skipping word clouds — install wordcloud: pip install wordcloud')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (lid, lname), col in zip(axes, LABEL_MAP.items(), COLORS):
        text = ' '.join(df[df[LABEL_COL] == lid][TEXT_COL].astype(str))
        wc = WordCloud(width=400, height=300, background_color='white',
                       colormap='RdYlGn', max_words=80).generate(text)
        ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
        ax.set_title(lname.capitalize(), fontsize=13)
    plt.suptitle('Word Clouds by Sentiment Class', y=1.02, fontsize=14)
    plt.tight_layout(); plt.show()

## 4. Preprocessing
Applied to classical ML only — the transformer receives raw text.

In [ ]:
_stop  = set(stopwords.words('english'))
_lemma = WordNetLemmatizer()

def preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r'http\S+|@\w+|#\w+|[^a-zA-Z\s]', ' ', text)
    return ' '.join(
        _lemma.lemmatize(t) for t in text.split()
        if t not in _stop and len(t) > 2
    )

df['clean_text'] = df[TEXT_COL].astype(str).apply(preprocess)

print('Before:', df[TEXT_COL].iloc[0])
print('After: ', df['clean_text'].iloc[0])

## 5. Train / Test Split

In [ ]:
X_raw   = df[TEXT_COL].astype(str).values    # raw text  → transformer
X_clean = df['clean_text'].values            # cleaned   → classical ML
y       = df[LABEL_COL].values

X_raw_tr, X_raw_te, X_cl_tr, X_cl_te, y_tr, y_te = train_test_split(
    X_raw, X_clean, y,
    test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {len(y_tr):,}  |  Test: {len(y_te):,}')

## 6. Classical ML
TF-IDF (unigrams + bigrams) with 5-fold CV across three classifiers; best model evaluated on the held-out test set.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=50_000, sublinear_tf=True)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1),
    'LinearSVC':           LinearSVC(max_iter=2000, C=1.0),
    'Naive Bayes':         MultinomialNB(alpha=0.1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, clf in classifiers.items():
    pipe   = Pipeline([('tfidf', tfidf), ('clf', clf)])
    scores = cross_val_score(pipe, X_cl_tr, y_tr, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:22s}  CV acc: {scores.mean():.4f} ± {scores.std():.4f}')

best_clf_name = max(cv_results, key=lambda k: cv_results[k].mean())
print(f'\nBest: {best_clf_name}')

In [ ]:
best_pipe = Pipeline([('tfidf', tfidf), ('clf', classifiers[best_clf_name])])
best_pipe.fit(X_cl_tr, y_tr)
y_pred_cl = best_pipe.predict(X_cl_te)

print(f'── {best_clf_name} ──')
print(classification_report(y_te, y_pred_cl, target_names=list(LABEL_MAP.values())))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_te, y_pred_cl), annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_MAP.values(), yticklabels=LABEL_MAP.values(), ax=ax)
ax.set(title=f'Confusion Matrix — {best_clf_name}', ylabel='True', xlabel='Predicted')
plt.tight_layout(); plt.show()

classical_acc = accuracy_score(y_te, y_pred_cl)

## 7. Transformer — Fine-tuning DistilBERT
Uses raw (un-cleaned) text. `BERT_SAMPLES` controls training size; set `None` to train on all data.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

class SentimentDataset(TorchDataset):
    def __init__(self, texts, labels):
        enc = tokenizer(list(texts), truncation=True, padding='max_length',
                        max_length=BERT_MAX_LEN, return_tensors='pt')
        self.ids    = enc['input_ids']
        self.mask   = enc['attention_mask']
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {'input_ids': self.ids[i], 'attention_mask': self.mask[i], 'labels': self.labels[i]}

n = BERT_SAMPLES or len(X_raw_tr)
train_ds = SentimentDataset(X_raw_tr[:n], y_tr[:n])
test_ds  = SentimentDataset(X_raw_te, y_te)

model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL, num_labels=len(LABEL_MAP)
).to(DEVICE)

training_args = TrainingArguments(
    output_dir='./bert_results',
    num_train_epochs=BERT_EPOCHS,
    per_device_train_batch_size=BERT_BATCH,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',      # use evaluation_strategy for transformers < 4.37
    save_strategy='no',
    logging_steps=50,
    report_to='none',
    fp16=(DEVICE == 'cuda'),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)
trainer.train()

In [ ]:
preds    = trainer.predict(test_ds)
y_pred_b = np.argmax(preds.predictions, axis=1)

print('── DistilBERT (fine-tuned) ──')
print(classification_report(y_te, y_pred_b, target_names=list(LABEL_MAP.values())))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_te, y_pred_b), annot=True, fmt='d', cmap='Purples',
            xticklabels=LABEL_MAP.values(), yticklabels=LABEL_MAP.values(), ax=ax)
ax.set(title='Confusion Matrix — DistilBERT', ylabel='True', xlabel='Predicted')
plt.tight_layout(); plt.show()

bert_acc = accuracy_score(y_te, y_pred_b)

## 8. Model Comparison

In [ ]:
cmp = pd.DataFrame({
    'Model':    [best_clf_name, 'DistilBERT (fine-tuned)'],
    'Accuracy': [classical_acc, bert_acc],
})

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(cmp['Model'], cmp['Accuracy'], color=['#3498db', '#9b59b6'])
for bar, val in zip(bars, cmp['Accuracy']):
    ax.text(bar.get_width() - 0.012, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', ha='right', color='white', fontweight='bold')
ax.set(xlim=(0, 1), xlabel='Test Accuracy', title='Classical ML vs Transformer')
plt.tight_layout(); plt.show()

print(cmp.to_string(index=False))

## 9. Inference on Custom Text

In [ ]:
def predict(text: str) -> dict:
    cl_pred = LABEL_MAP[best_pipe.predict([preprocess(text)])[0]]
    enc = tokenizer(text, return_tensors='pt', truncation=True,
                    max_length=BERT_MAX_LEN, padding='max_length').to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits
    bt_pred = LABEL_MAP[logits.argmax(-1).item()]
    return {'text': text[:60], 'classical': cl_pred, 'bert': bt_pred}

examples = [
    'This product is absolutely fantastic, I love every bit of it!',
    'It arrived on time and does exactly what it says.',
    'Completely broken out of the box. Total waste of money.',
]

pd.DataFrame([predict(t) for t in examples])